# Armenian Syllable Counter — Inspection Notebook

This notebook exposes the internal syllable-counting functions used by `QuantitativeLinguisticsAnalyzer`
in `linguistics/metrics/text_metrics.py` so you can inspect outputs and verify correctness on real
Western Armenian words.

Two functions are demonstrated:
- `count_syllables(word)` — fast vowel-nucleus counter (`linguistics.morphology.core`)
- `count_syllables_with_context(word)` — context-aware version that includes hidden ə (`linguistics.morphology.difficulty`)

The Flesch-Kincaid grade-level metric uses `count_syllables`.

In [ ]:
import sys, os
# Add the project root to the path so the package is importable from this notebook.
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)

In [ ]:
from linguistics.morphology.core import count_syllables, VOWELS, DIGRAPH_U
from linguistics.morphology.difficulty import count_syllables_with_context

print("count_syllables imported ✓")
print("count_syllables_with_context imported ✓")

## Vowel Set & Digraph

These are the characters that `count_syllables` recognises as vowel nuclei.
The `ու` digraph (ո + ւ) is handled separately — it counts as **one** syllable, not two.

In [ ]:
print("Armenian vowels (VOWELS set):")
for ch in sorted(VOWELS, key=ord):
    print(f"  U+{ord(ch):04X}  {ch}")

print()
print(f"DIGRAPH_U  =  '{DIGRAPH_U}'  (ո + ւ → /u/, counts as 1 syllable)")

## Test on a Word List

Expected counts are noted in comments.  
Words are from the Western Armenian lexicon (classical orthography).

In [ ]:
# (word, expected_syllables, notes)
test_words = [
    # Monosyllables
    ("մարդ",  1, "mart — man"),
    ("բան",   1, "pan  — thing"),
    ("գիր",   1, "gir  — letter/book"),
    # Bisyllables
    ("հայոց", 2, "hayots — Armenian (genitive)"),
    ("մայրիկ",2, "mayrig — mom"),
    ("կեանք", 2, "gyank  — life"),
    # Trisyllables
    ("հայրենիք", 3, "hayrenig — homeland"),
    ("ազատութիւն", 4, "azadoutyun — freedom"),
    # ու digraph — must count as ONE vowel
    ("ուրախ", 2, "ourakh — happy (ու = 1 vowel nucleus)"),
    ("սիրուն", 2, "siroun — beautiful"),
    ("կու",   1, "gu    — (auxiliary, 1 syllable)"),
    # Classical diphthongs
    ("ոյժ",   1, "voyzh  — power (ո before ւ, classical)"),
    ("լոյս",  2, "louys  — light"),
]

header = f"{'Word':<16} {'Exp':>4} {'count_syllables':>16} {'with_context':>13}  Notes"
print(header)
print("-" * len(header))

for word, expected, notes in test_words:
    basic  = count_syllables(word)
    ctx    = count_syllables_with_context(word)
    match  = "✓" if basic == expected else "✗"
    print(f"{word:<16} {expected:>4} {match} {basic:>13}  {ctx:>12}  {notes}")

## Flesch-Kincaid in Context

The `QuantitativeLinguisticsAnalyzer._compute_syntactic_metrics()` method uses:

```python
from linguistics.morphology.core import count_syllables as _count_syl
total_syllables = sum(_count_syl(w) for w in words)
avg_syllables_per_word = total_syllables / total_words
fk_grade = 0.39 * asl + 11.8 * avg_syllables_per_word - 15.59
```

Below you can test FK on a short passage.

In [ ]:
from cleaning.armenian_tokenizer import extract_words
import re

sample_text = "Հայաստան երկիրն է Կովկասում։ Ան ունի հին մշակոյթ եւ գեղեցիկ բնութիւն։"

words = extract_words(sample_text, min_length=1)
sentences = [s.strip() for s in re.split(r'[։?!:]+', sample_text) if s.strip()]

total_words = len(words)
num_sentences = max(len(sentences), 1)
total_syllables = sum(count_syllables(w) for w in words)

asl = total_words / num_sentences
avg_syl = total_syllables / total_words if total_words else 2.0
fk = 0.39 * asl + 11.8 * avg_syl - 15.59

print(f"Words:          {total_words}")
print(f"Sentences:      {num_sentences}")
print(f"Total syllables:{total_syllables}")
print(f"ASL:            {asl:.2f}")
print(f"Avg syl/word:   {avg_syl:.2f}")
print(f"FK grade:       {fk:.2f}")

## Interactive: Test Any Word

Edit `my_word` in the cell below and run it to see the syllable breakdown.

In [ ]:
my_word = "ազատութիւն"  # ← edit this

basic = count_syllables(my_word)
ctx   = count_syllables_with_context(my_word)

# Show character-by-character analysis
print(f"Word: {my_word}")
print()
print("Character breakdown:")
i = 0
while i < len(my_word):
    if i + 1 < len(my_word) and my_word[i:i+2] == DIGRAPH_U:
        print(f"  [{i}] '{my_word[i:i+2]}' → DIGRAPH ու  (1 vowel nucleus)")
        i += 2
    elif my_word[i] in VOWELS:
        print(f"  [{i}] '{my_word[i]}'  → VOWEL")
        i += 1
    else:
        print(f"  [{i}] '{my_word[i]}'  → consonant")
        i += 1

print()
print(f"count_syllables()              → {basic}")
print(f"count_syllables_with_context() → {ctx}")